<a href="https://colab.research.google.com/github/RonakkudalAI/Deep-Neural-Network/blob/main/02_sales_text_classification_with_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Classification with Transformers

## Sales case study: routing customer messages to the right sales workflow

A B2B software company receives many inbound messages from website forms, emails, LinkedIn campaigns, and chat. The sales team wants to automatically route each message to the right workflow.

In this notebook, we will build an end-to-end text classification system using Hugging Face.

The model will classify messages into:

- Demo Request
- Pricing Inquiry
- Product Question
- Renewal Interest
- Complaint
- Not Relevant

The business goal is to reduce manual sorting and help sales teams respond faster.


## Learning roadmap

1. Understand the sales routing problem.
2. Create a realistic inbound sales message dataset.
3. Explore message categories.
4. Understand text classification.
5. Compare sentiment analysis and text classification.
6. Learn tokenization for transformer models.
7. Fine-tune a transformer classifier.
8. Evaluate class-level performance.
9. Predict categories for new messages.
10. Convert predicted categories into sales actions.


## Code microscope: important Hugging Face words before we train

Before we run the model, let us slow down and understand the objects that will appear in the code.

| Term | Simple meaning | Why it matters |
|---|---|---|
| `model_checkpoint` | The name or path of a pretrained model. | It tells Hugging Face which model files to download or load. |
| Checkpoint | A saved model state containing learned weights, configuration, and sometimes tokenizer files. | We do not train from zero. We start from a model that already learned English patterns. |
| `AutoTokenizer` | A helper that automatically loads the correct tokenizer for the checkpoint. | Different models use different tokenization rules, so we should not guess manually. |
| Tokenizer | Converts text into token IDs and attention masks. | Neural networks read numbers, not raw words. |
| `AutoModelForSequenceClassification` | A helper that loads a transformer model with a classification layer on top. | We need one final label for each full message. |
| Sequence classification | Classifying an entire text sequence into one label. | Sentiment and sales routing are sequence classification tasks. |
| `logits` | Raw model scores before probabilities. | We use them to choose the class with the highest score. |
| `labels` | Correct answers from the dataset. | The Trainer compares predictions with labels during evaluation. |
| `np.argmax` | Selects the position of the largest value. | The largest logit becomes the predicted class. |
| `TrainingArguments` | A configuration object for training settings. | It controls epochs, batch size, learning rate, evaluation timing, and logging. |
| `Trainer` | Hugging Face's training manager. | It runs the training loop without us manually writing every PyTorch step. |


## 1. Why text classification matters in sales

Sales teams lose time when every message must be manually read and routed.

Example:

- A demo request should go to an account executive.
- A pricing question should go to sales development or revenue operations.
- A complaint should go to customer support.
- A renewal message should go to customer success.

Text classification helps route messages automatically.


## 2. Simple explanation

Text classification means assigning a category to a piece of text.

Real-world analogy:

Imagine a mailroom where incoming letters are placed into different trays:

- Sales
- Support
- Billing
- HR
- Marketing

Our model is a smart mailroom assistant for sales messages.


## 3. Tiny example

| Message | Category |
|---|---|
| "Can we schedule a demo next week?" | Demo Request |
| "How much does the enterprise plan cost?" | Pricing Inquiry |
| "Your dashboard is not loading for our team." | Complaint |

The model learns from labeled examples and predicts a category for new messages.


In [ ]:
# This cell installs the main libraries used in this notebook.
# Run it once if you are using a fresh environment such as Google Colab.
# If the libraries are already installed, this cell may finish quickly.
%pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy matplotlib seaborn torch


In [ ]:
# pandas helps us create and analyze tabular datasets.
import pandas as pd

# numpy helps with numerical operations.
import numpy as np

# matplotlib is used for basic charts.
import matplotlib.pyplot as plt

# seaborn makes charts easier to read.
import seaborn as sns

# torch is the deep learning library used by Hugging Face models.
import torch

# train_test_split separates our data into training and testing parts.
from sklearn.model_selection import train_test_split

# LabelEncoder converts text labels such as "Positive" into numbers such as 2.
from sklearn.preprocessing import LabelEncoder

# classification_report gives precision, recall, F1-score, and accuracy.
from sklearn.metrics import classification_report

# confusion_matrix shows where the model is correct and where it is confused.
from sklearn.metrics import confusion_matrix

# Dataset is a Hugging Face object used by the Trainer API.
from datasets import Dataset

# AutoTokenizer loads the correct tokenizer for a selected model name.
from transformers import AutoTokenizer

# AutoModelForSequenceClassification loads a transformer model for classification.
from transformers import AutoModelForSequenceClassification

# TrainingArguments stores training settings such as epochs and batch size.
from transformers import TrainingArguments

# Trainer handles the training loop for transformer models.
from transformers import Trainer

# set_seed makes the experiment easier to reproduce.
from transformers import set_seed

# We set a seed so that the train-test split and model training are more stable.
set_seed(42)


## 4. Create a realistic sales dataset

Each row represents one inbound message received by the company.

Columns:

- `message_id`: unique message number
- `source`: where the message came from
- `message`: customer or prospect text
- `category`: the routing label


In [ ]:
# We create a dataset of inbound sales messages.
sales_records = [
    {"message_id": 1, "source": "Website Form", "message": "Can your team show us a demo of the CRM platform next Tuesday?", "category": "Demo Request"},
    {"message_id": 2, "source": "Email", "message": "Please share pricing for fifty users on the annual plan.", "category": "Pricing Inquiry"},
    {"message_id": 3, "source": "Chat", "message": "Does your product integrate with Salesforce and HubSpot?", "category": "Product Question"},
    {"message_id": 4, "source": "Email", "message": "Our contract ends next quarter and we want to discuss renewal options.", "category": "Renewal Interest"},
    {"message_id": 5, "source": "Chat", "message": "The dashboard keeps freezing and our sales managers are blocked.", "category": "Complaint"},
    {"message_id": 6, "source": "Website Form", "message": "I am looking for jobs in your company.", "category": "Not Relevant"},
    {"message_id": 7, "source": "LinkedIn", "message": "We would like a product walkthrough for our regional sales team.", "category": "Demo Request"},
    {"message_id": 8, "source": "Website Form", "message": "Is there a discount if we buy licenses for two hundred users?", "category": "Pricing Inquiry"},
    {"message_id": 9, "source": "Email", "message": "Can your lead scoring model work with custom fields?", "category": "Product Question"},
    {"message_id": 10, "source": "Email", "message": "We are happy with the product and want to renew for another year.", "category": "Renewal Interest"},
    {"message_id": 11, "source": "Chat", "message": "The report export failed during our board meeting preparation.", "category": "Complaint"},
    {"message_id": 12, "source": "Website Form", "message": "Please unsubscribe me from your newsletter.", "category": "Not Relevant"},
    {"message_id": 13, "source": "Website Form", "message": "Book a demo for our sales operations team this Friday.", "category": "Demo Request"},
    {"message_id": 14, "source": "LinkedIn", "message": "What is the monthly cost for the professional package?", "category": "Pricing Inquiry"},
    {"message_id": 15, "source": "Chat", "message": "Does the mobile app support offline meeting notes?", "category": "Product Question"},
    {"message_id": 16, "source": "Email", "message": "We need to add more seats before renewing the contract.", "category": "Renewal Interest"},
    {"message_id": 17, "source": "Email", "message": "Support has not responded and the pipeline view is still broken.", "category": "Complaint"},
    {"message_id": 18, "source": "Website Form", "message": "I accidentally submitted this form while browsing.", "category": "Not Relevant"},
    {"message_id": 19, "source": "Chat", "message": "Can someone walk us through the forecasting module?", "category": "Demo Request"},
    {"message_id": 20, "source": "Email", "message": "Send a quote for the enterprise security add-on.", "category": "Pricing Inquiry"},
    {"message_id": 21, "source": "Website Form", "message": "Can managers approve discounts inside your tool?", "category": "Product Question"},
    {"message_id": 22, "source": "Email", "message": "We want to extend our subscription for the next fiscal year.", "category": "Renewal Interest"},
    {"message_id": 23, "source": "Chat", "message": "The login page is down and my sales team cannot access accounts.", "category": "Complaint"},
    {"message_id": 24, "source": "Website Form", "message": "Do you sell office chairs?", "category": "Not Relevant"},
    {"message_id": 25, "source": "LinkedIn", "message": "I want to see how your CRM handles enterprise account planning.", "category": "Demo Request"},
    {"message_id": 26, "source": "Email", "message": "Do you charge separately for implementation and onboarding?", "category": "Pricing Inquiry"},
    {"message_id": 27, "source": "Chat", "message": "Is role-based access control included in the standard plan?", "category": "Product Question"},
    {"message_id": 28, "source": "Email", "message": "Our procurement team is ready to renew if the terms stay the same.", "category": "Renewal Interest"},
    {"message_id": 29, "source": "Chat", "message": "The email sync has been unreliable for three days.", "category": "Complaint"},
    {"message_id": 30, "source": "Website Form", "message": "This message is a test submission from our IT team.", "category": "Not Relevant"},
]

# We convert the list into a DataFrame.
sales_df = pd.DataFrame(sales_records)

# We inspect the first rows.
sales_df.head()


## 5. Explore the dataset

Data exploration helps us understand whether the classes are balanced and whether messages have enough useful text.


In [ ]:
# We check the number of rows and columns.
sales_df.shape


In [ ]:
# We count messages in each category.
sales_df["category"].value_counts()


In [ ]:
# We calculate message length in words.
sales_df["word_count"] = sales_df["message"].str.split().str.len()

# We show word count statistics.
sales_df["word_count"].describe()


In [ ]:
# We set the chart size.
plt.figure(figsize=(10, 4))

# We plot the number of messages per category.
sns.countplot(data=sales_df, x="category", order=sales_df["category"].value_counts().index)

# We rotate x-axis labels so they do not overlap.
plt.xticks(rotation=30, ha="right")

# We add labels and title.
plt.title("Inbound sales messages by category")
plt.xlabel("Message category")
plt.ylabel("Number of messages")

# We display the chart.
plt.show()


## 6. Text classification versus sentiment analysis

Sentiment analysis asks: "What is the emotional tone?"

Text classification asks: "Which business category does this text belong to?"

Example:

"The product looks useful. Can you share pricing?"

Sentiment: Positive or neutral.

Sales category: Pricing Inquiry.

For sales routing, category is more useful than emotion.


## 7. Why use a transformer model?

Older models often rely heavily on exact words. Transformers understand context better.

Example:

"Can we see the platform before buying?"

The message does not contain the word "demo", but it still means Demo Request.

A transformer can learn that "see the platform" and "demo" are related in this business context.


## 8. Prepare labels

The model needs numeric labels. We convert text categories into numbers.


In [ ]:
# We create the label encoder.
label_encoder = LabelEncoder()

# We convert category names into numeric labels.
sales_df["label"] = label_encoder.fit_transform(sales_df["category"])

# We build a readable mapping of category to number.
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

# We display the mapping.
label_mapping


In [ ]:
# We keep the message and label columns for modeling.
model_df = sales_df[["message", "label"]].copy()

# We rename message to text for Hugging Face consistency.
model_df = model_df.rename(columns={"message": "text"})

# We display the prepared data.
model_df.head()


## 9. Train-test split

We split the data into training and testing sets. The model learns from training data and is evaluated on test data.


In [ ]:
# We split the data while keeping category distribution similar.
train_df, test_df = train_test_split(
    model_df,
    test_size=0.25,
    random_state=42,
    stratify=model_df["label"],
)

# We reset the row index for training data.
train_df = train_df.reset_index(drop=True)

# We reset the row index for test data.
test_df = test_df.reset_index(drop=True)

# We print both dataset sizes.
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))


## 10. Tokenization

Tokenization converts text into model-readable numbers.

Transformer flow:

Text -> Tokens -> Token IDs -> Transformer model -> Category prediction


In [ ]:
# A checkpoint is the name or location of saved pretrained model files.
# Here, "distilbert-base-uncased" is a public model checkpoint on the Hugging Face Hub.
# "distilbert" means it is a smaller, faster version inspired by BERT.
# "base" means it is the standard-size DistilBERT model.
# "uncased" means the model treats "Bank", "bank", and "BANK" mostly the same by lowercasing text.
model_checkpoint = "distilbert-base-uncased"

# AutoTokenizer is a smart loader.
# Instead of manually choosing a tokenizer class, we give Hugging Face the checkpoint name.
# Hugging Face reads the checkpoint configuration and loads the tokenizer that matches this model.
# This matters because BERT, DistilBERT, RoBERTa, and T5 may tokenize text differently.
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# We choose one real sales message from our finance dataset.
# loc[0, "message"] means: row index 0, column named "message".
sample_text = sales_df.loc[0, "message"]

# The tokenizer converts text into a dictionary of model inputs.
# The most important keys are usually:
# - input_ids: numeric IDs for tokens.
# - attention_mask: 1 for real tokens and 0 for padding tokens.
sample_tokens = tokenizer(sample_text)

# We display the original human-readable text.
print("Original text:")
print(sample_text)

# input_ids are the numbers the model will actually receive.
# Each number points to a token in the tokenizer vocabulary.
print("\nToken IDs:")
print(sample_tokens["input_ids"])

# convert_ids_to_tokens changes the numeric IDs back into token strings.
# This helps students see how the sentence was broken into smaller pieces.
print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(sample_tokens["input_ids"]))


In [ ]:
# Hugging Face Trainer works best with Hugging Face Dataset objects.
# Dataset.from_pandas converts our pandas DataFrame into that format.
train_dataset = Dataset.from_pandas(train_df)

# We convert the test DataFrame in the same way.
test_dataset = Dataset.from_pandas(test_df)

# This function receives a batch, not just one row.
# A batch is a small group of examples processed together for speed.
def tokenize_batch(batch):
    # batch["text"] contains several sales messages from the dataset.
    # tokenizer(...) converts those messages into input_ids and attention_mask.
    return tokenizer(
        batch["text"],

        # padding="max_length" forces every example to have the same length.
        # Models train faster when examples in a batch have equal shape.
        padding="max_length",

        # truncation=True cuts text that is longer than max_length.
        # Without truncation, very long text can exceed the model limit and cause an error.
        truncation=True,

        # max_length=96 means every sales message becomes 96 token positions.
        # Short messages are padded. Long messages are cut to 96 tokens.
        max_length=96,
    )

# map applies tokenize_batch to the full training dataset.
# batched=True means Hugging Face sends multiple rows at once to the function.
tokenized_train = train_dataset.map(tokenize_batch, batched=True)

# We apply the exact same tokenization process to the test dataset.
tokenized_test = test_dataset.map(tokenize_batch, batched=True)

# with_format("torch") tells the dataset to return PyTorch tensors.
# PyTorch tensors are the numeric format expected by the transformer model.
tokenized_train = tokenized_train.with_format("torch")

# We also format the test dataset as PyTorch tensors.
tokenized_test = tokenized_test.with_format("torch")


## 11. Load and fine-tune the model

We use the same model family as the sentiment notebook, but the labels are different. This is a multi-class routing problem.


In [ ]:
# num_labels tells the model how many sales categories it must predict.
# In this notebook, categories include Demo Request, Pricing Inquiry, Product Question, and others.
num_labels = sales_df["label"].nunique()

# AutoModelForSequenceClassification loads a transformer model for full-message classification.
# It loads the base DistilBERT language model and adds a classification layer for our sales categories.
model = AutoModelForSequenceClassification.from_pretrained(
    # model_checkpoint tells Hugging Face which pretrained model to load.
    model_checkpoint,

    # num_labels controls the size of the final output layer.
    # If we have 6 categories, the model produces 6 raw scores for every message.
    num_labels=num_labels,
)


In [ ]:
# Trainer calls this function during evaluation.
# eval_prediction is created by Trainer after the model predicts on the evaluation dataset.
def compute_metrics(eval_prediction):
    # eval_prediction contains two main things:
    # 1. predictions: raw model outputs, usually called logits.
    # 2. label_ids: the correct labels from the dataset.
    logits, labels = eval_prediction

    # logits are raw scores produced by the model before softmax.
    # Example for one message with three classes: [1.2, -0.4, 2.8]
    # These are not probabilities yet, but the largest score usually represents the predicted class.

    # np.argmax(logits, axis=-1) finds the index of the largest logit for each row.
    # axis=-1 means "look across the class scores for each example".
    # If logits are [1.2, -0.4, 2.8], argmax returns 2.
    predictions = np.argmax(logits, axis=-1)

    # labels are the correct answer IDs from our test/evaluation data.
    # predictions == labels creates True/False values for correct and incorrect predictions.
    # mean() converts True to 1 and False to 0, then calculates the average correctness.
    accuracy = (predictions == labels).mean()

    # Trainer expects a dictionary of metric names and metric values.
    # We return accuracy so it appears in the training logs.
    return {"accuracy": accuracy}


In [ ]:
# TrainingArguments stores training settings for Trainer.
# Think of it as the training configuration form.
training_args = TrainingArguments(
    # output_dir is where Trainer writes logs and temporary output files.
    output_dir="./sales_text_classification_model",

    # num_train_epochs is the number of complete passes over the training dataset.
    # More epochs can improve learning, but too many can cause memorization on small data.
    num_train_epochs=2,

    # per_device_train_batch_size controls how many messages are processed together during training.
    # A value of 4 is small and laptop-friendly.
    per_device_train_batch_size=4,

    # per_device_eval_batch_size controls how many messages are processed together during evaluation.
    per_device_eval_batch_size=4,

    # learning_rate controls the size of each model update.
    # For transformer fine-tuning, small values such as 2e-5 are common.
    learning_rate=2e-5,

    # eval_strategy="epoch" means evaluate after each epoch.
    eval_strategy="epoch",

    # save_strategy="no" avoids creating model checkpoint folders during this beginner lab.
    save_strategy="no",

    # logging_steps controls how often progress messages appear.
    logging_steps=5,

    # report_to="none" prevents external logging integrations from asking for credentials.
    report_to="none",
)

# Trainer manages the training loop.
trainer = Trainer(
    # The transformer classifier we want to fine-tune.
    model=model,

    # The training settings above.
    args=training_args,

    # Tokenized training examples used for learning.
    train_dataset=tokenized_train,

    # Tokenized test examples used for evaluation.
    eval_dataset=tokenized_test,

    # The tokenizer that created input_ids and attention_mask.
    tokenizer=tokenizer,

    # The metric function that converts logits into accuracy.
    compute_metrics=compute_metrics,
)

# This starts fine-tuning for the sales routing task.
trainer.train()


## 12. Evaluation

In sales routing, class-level metrics matter.

Example:

If the model often confuses Complaint with Product Question, customers may not receive support quickly enough.


In [ ]:
# trainer.predict runs the model on test examples.
# It does not update weights; it only produces predictions.
test_predictions = trainer.predict(tokenized_test)

# logits are raw class scores from the model.
# For 6 sales categories, each message receives 6 logits.
logits = test_predictions.predictions

# np.argmax chooses the category with the highest score for each message.
predicted_label_ids = np.argmax(logits, axis=1)

# true_label_ids are the correct category numbers from the test set.
true_label_ids = test_df["label"].values

# Convert predicted numeric labels into readable category names.
predicted_labels = label_encoder.inverse_transform(predicted_label_ids)

# Convert true numeric labels into readable category names.
true_labels = label_encoder.inverse_transform(true_label_ids)

# Print precision, recall, F1-score, and accuracy for the sales categories.
print(classification_report(true_labels, predicted_labels))


In [ ]:
# We create a confusion matrix.
cm = confusion_matrix(true_labels, predicted_labels, labels=label_encoder.classes_)

# We set chart size.
plt.figure(figsize=(9, 5))

# We draw a heatmap.
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    cmap="Greens",
)

# We rotate x-axis labels for readability.
plt.xticks(rotation=30, ha="right")

# We add labels and title.
plt.xlabel("Predicted category")
plt.ylabel("Actual category")
plt.title("Confusion matrix for sales message classification")

# We show the chart.
plt.show()


## 13. Predict category for new messages

Now we create a function that can classify new inbound messages.


In [ ]:
# This function predicts the sales category for one new message.
def predict_sales_category(message):
    # message is the raw inbound text from a website form, email, chat, or LinkedIn.

    # The tokenizer prepares the message for the transformer model.
    # return_tensors="pt" returns PyTorch tensors.
    # max_length=96 keeps the input size fixed and manageable.
    encoded_input = tokenizer(
        message,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=96,
    )

    # Move all input tensors to the same device as the model.
    encoded_input = {key: value.to(model.device) for key, value in encoded_input.items()}

    # Prediction does not need gradients, so no_grad saves memory and time.
    with torch.no_grad():
        # The model returns logits, which are raw scores for each sales category.
        output = model(**encoded_input)

    # Convert logits into probabilities for easier interpretation.
    probabilities = torch.softmax(output.logits, dim=1).cpu().numpy()[0]

    # Choose the category with the highest probability.
    predicted_id = int(np.argmax(probabilities))

    # Convert the numeric category ID back to the original category name.
    predicted_category = label_encoder.inverse_transform([predicted_id])[0]

    # Store every category probability in a readable dictionary.
    probability_table = {
        label_encoder.inverse_transform([class_id])[0]: float(probability)
        for class_id, probability in enumerate(probabilities)
    }

    # Return both the category and the probability details.
    return predicted_category, probability_table


In [ ]:
# We create new sales messages for testing.
new_sales_messages = [
    "Can someone show our leadership team how the forecasting dashboard works?",
    "Please send the cost for 120 users with onboarding included.",
    "The system has been unavailable and our reps cannot update deals.",
    "We are reviewing our subscription and may continue for another year.",
]

# We classify each new message.
for message in new_sales_messages:
    # We call the prediction function.
    category, probabilities = predict_sales_category(message)

    # We print the message.
    print("Message:", message)

    # We print the predicted category.
    print("Predicted category:", category)

    # We print the probabilities.
    print("Probabilities:", probabilities)

    # We print a separator.
    print("-" * 80)


## 14. Convert categories into business actions

The model should not stop at prediction. It should help the sales organization act.


In [ ]:
# This function maps predicted categories to business owners.
def recommend_sales_action(category):
    # Demo requests should go to account executives.
    if category == "Demo Request":
        return "Assign to account executive for demo scheduling"

    # Pricing inquiries should go to the sales development or pricing team.
    if category == "Pricing Inquiry":
        return "Send to sales development team with pricing playbook"

    # Product questions should go to a solution consultant.
    if category == "Product Question":
        return "Route to solution consultant"

    # Renewal interest should go to customer success.
    if category == "Renewal Interest":
        return "Notify customer success manager"

    # Complaints should go to support.
    if category == "Complaint":
        return "Create support ticket and alert customer success"

    # Not relevant messages can be filtered.
    return "Mark as not relevant for sales queue"

# We prepare an empty list for routed outputs.
routed_messages = []

# We process every new sales message.
for message in new_sales_messages:
    # We predict the category.
    category, probabilities = predict_sales_category(message)

    # We recommend an action.
    action = recommend_sales_action(category)

    # We append the result.
    routed_messages.append(
        {
            "message": message,
            "predicted_category": category,
            "recommended_action": action,
        }
    )

# We convert results to a DataFrame.
routed_messages_df = pd.DataFrame(routed_messages)

# We display routed messages.
routed_messages_df


## 15. Limitations and risks

Possible issues:

- A short message such as "Need help" may be unclear.
- One message may contain multiple intents.
- The model may confuse Product Question and Demo Request.
- Business language changes over time, so the model must be updated.
- High-value accounts may need human review even if the model is confident.


## 16. Alternatives

Other approaches include:

- Keyword routing rules.
- TF-IDF with logistic regression.
- Sentence embeddings with nearest-neighbor search.
- Zero-shot classification using a larger transformer.
- LLM-based routing with careful governance.

## Student practice

1. Add a new class called `Security Review`.
2. Add 20 more realistic sales messages.
3. Test messages that contain two intents.
4. Compare model output with simple keyword rules.
5. Design an escalation rule for enterprise customers.

## Final takeaway

You built an end-to-end sales text classification system that routes inbound messages to the correct sales workflow.
